# Feedback on Learning Programs

In [1]:
# data manipulation
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# statistical tests
from scipy import stats

# statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

# logistic regression diagnostics
from statsmodels.stats.outliers_influence import variance_inflation_factor

# sklearn metrics
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

# display settings
pd.set_option("display.max_columns", None)

# plotting style
sns.set_style("whitegrid")

In [2]:
# load learning dataset
url = "https://raw.githubusercontent.com/keithmcnulty/peopleanalytics-regression-book/master/data/learning.csv"

learning_df = pd.read_csv(url)

# basic inspection
print(learning_df.shape)
print("\nColumns:")
print(learning_df.columns)

learning_df.head()

(4974, 8)

Columns:
Index(['idcode', 'rec', 'rel', 'fun', 'clar', 'home', 'class', 'fac'], dtype='object')


,idcode,rec,rel,fun,clar,home,class,fac
0,176304,0,3.0,5.0,3.0,4.0,4.0,4.0
1,176304,0,4.0,4.0,5.0,4.0,4.0,1.0
2,176304,0,3.0,4.0,4.0,3.0,4.0,1.0
3,176304,1,4.0,5.0,5.0,4.0,5.0,5.0
4,176304,0,3.0,5.0,5.0,4.0,5.0,4.0


Use a standard **binomial logistic regression model** predicting rec from the learning-experience ratings. This would identify which program-quality dimensions are associated with higher odds of recommendation, while treating each survey response as independent.

The **hierarchical model**should help identify which feedback dimensions are most associated with recommendation likelihood after accounting for participant-level differences in baseline recommendation tendency. This allows us to estimate whether factors such as relevance, fun, clarity, applicability, class experience, or facilities predict recommendation while recognizing that repeated responses from the same participant are not independent.

The hierarchy is that individual feedback records are nested within participants. The dataset contains 4,974 feedback instances from 326 participants, meaning each participant may contribute multiple rows. Because responses from the same participant are not independent, the model should account for participant-level clustering or participant-level random effects.

**Exercise Workflow — Learning Program Recommendation Analysis**
1. Business / Research Framing
2. Variable Classification
3. Exploratory Data Analysis
4. Hierarchy / Repeated-Records Check
5. Baseline Logistic Regression
6. Baseline Logistic Diagnostics
7. Hierarchical Logistic Regression
8. Compare Baseline vs Hierarchical Model
9. Interpretation Framework
10. Final Conclusions and Recommendations

**1. Business / Research Framing**

Primary question:

* Which aspects of the learning program experience are most strongly associated with whether a participant recommends the program?

Secondary modeling issue:

* Because the same participant may provide feedback on multiple programs, how should we account for repeated observations from the same person?

**2. Variable Classification**

| Variable | Type                   | Role                                        |
| -------- | ---------------------- | ------------------------------------------- |
| `rec`    | binary 0/1             | outcome: recommended or not                 |
| `rel`    | ordinal rating 1–5     | relevance predictor                         |
| `fun`    | ordinal rating 1–5     | engagement/enjoyment predictor              |
| `clar`   | ordinal rating 1–5     | clarity predictor                           |
| `home`   | ordinal rating 1–5     | applicability/homework/usefulness predictor |
| `class`  | ordinal rating 1–5     | class/session quality predictor             |
| `fac`    | ordinal rating 1–5     | facility/environment predictor              |
| `idcode` | participant identifier | grouping / hierarchy variable               |


**3. Exploratory Data Analysis**

Initial EDA:

* inspect shape, columns, and data types
* check missing values
* inspect distribution of rec
* inspect distributions of all 1–5 rating predictors
* calculate recommendation rate overall
* calculate recommendation rate by each rating level for each predictor
* examine correlations among predictors
* check whether predictors are highly collinear

Key EDA takeaway:

* The dataset has repeated feedback records nested within participants and nontrivial missingness in the predictor ratings. This supports the need for a hierarchical model and requires a clear missing-data decision before modeling.

In [3]:
# inspect data types
learning_df.dtypes

,0
idcode,int64
rec,int64
rel,float64
fun,float64
clar,float64
home,float64
class,float64
fac,float64


In [4]:
# check missing values
learning_df.isnull().sum()

,0
idcode,0
rec,0
rel,124
fun,186
clar,218
home,427
class,638
fac,200


In [5]:
# summary statistics
learning_df.describe()

,idcode,rec,rel,fun,clar,home,class,fac
count,4974.000000,4974.000000,4850.000000,4788.000000,4756.000000,4547.000000,4336.000000,4774.000000
mean,176578.235625,0.413148,3.453402,3.851921,3.540370,3.699802,3.148293,3.025555
std,155.574287,0.492449,0.821194,0.777264,0.838512,0.802168,0.895413,0.898457
min,176304.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,176456.000000,0.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
50%,176579.000000,0.000000,3.000000,4.000000,4.000000,4.000000,3.000000,3.000000
75%,176701.000000,1.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000
max,176850.000000,1.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


In [6]:
# outcome distribution
learning_df["rec"].value_counts()

,count
rec,
0,2919
1,2055


In [7]:
# outcome proportions
learning_df["rec"].value_counts(normalize=True)

,proportion
rec,
0,0.586852
1,0.413148


In [8]:
# number of unique participants
learning_df["idcode"].nunique()

326

In [9]:
# records per participant
records_per_participant = learning_df["idcode"].value_counts()

records_per_participant.describe()

,count
count,326.000000
mean,15.257669
std,4.993644
min,5.000000
25%,10.000000
50%,16.000000
75%,20.000000
max,22.000000


In [10]:
# how many participants appear more than once
(records_per_participant > 1).sum()

np.int64(326)

In [11]:
# distribution of records per participant
records_per_participant.value_counts().sort_index()

,count
count,
5,4
6,19
7,4
8,4
9,17
10,55
14,21
15,15
16,34


In [12]:
# distributions of rating predictors
rating_cols = ["rel", "fun", "clar", "home", "class", "fac"]

for col in rating_cols:
    print(f"\n{col} distribution")
    print(learning_df[col].value_counts().sort_index())


rel distribution
rel
1.0      61
2.0     374
3.0    2197
4.0    1741
5.0     477
Name: count, dtype: int64

fun distribution
fun
1.0      28
2.0     127
3.0    1304
4.0    2396
5.0     933
Name: count, dtype: int64

clar distribution
clar
1.0      66
2.0     307
3.0    1950
4.0    1857
5.0     576
Name: count, dtype: int64

home distribution
home
1.0      27
2.0     160
3.0    1708
4.0    1908
5.0     744
Name: count, dtype: int64

class distribution
class
1.0     165
2.0     694
3.0    2085
4.0    1117
5.0     275
Name: count, dtype: int64

fac distribution
fac
1.0     273
2.0     818
3.0    2427
4.0    1026
5.0     230
Name: count, dtype: int64


**4. Hierarchy / Repeated-Records Check**

The data have a clear repeated-measures structure: multiple feedback records are nested within each participant. Therefore, observations are not fully independent, and a standard logistic regression may underestimate uncertainty because it treats all 4,974 records as independent.

**5. Baseline Logistic Regression**

In [18]:
# define model variables
model_vars = ["rec", "rel", "fun", "clar", "home", "class", "fac"]

# create complete-case dataframe for baseline logistic regression
learning_model_df = learning_df[model_vars].dropna().copy()

In [19]:
# define model variables
model_vars = ["rec", "rel", "fun", "clar", "home", "class", "fac"]

# create complete-case dataframe for baseline logistic regression
learning_model_df = learning_df[model_vars].dropna().copy()

print("Original rows:", learning_df.shape[0])
print("Complete-case rows:", learning_model_df.shape[0])
print("Rows dropped:", learning_df.shape[0] - learning_model_df.shape[0])

Original rows: 4974
Complete-case rows: 4120
Rows dropped: 854


**Baseline Logistic Regression** Results

Recommendation likelihood is most strongly associated with perceived relevance, followed by clarity, class quality, and faculty/facility quality. However, this model ignores participant-level clustering, so the standard errors and significance tests may be too optimistic.

Key results:

* rel is the strongest positive predictor. A one-point increase in relevance rating is associated with about 2.72 times the odds of recommending the program, holding other ratings constant.

* clar is also positive and significant. A one-point increase in clarity is associated with about 1.70 times the odds of recommending.

* class is positive and significant. A one-point increase in class quality is associated with about 1.62 times the odds of recommending.

* fac is positive and significant. A one-point increase in facilities/instructor quality is associated with about 1.44 times the odds of recommending.

* home is negative and significant. A one-point increase in the homework/project rating is associated with about 22% lower odds of recommending, holding the other predictors constant.

* fun is negative but not statistically significant.

In [20]:
# baseline logistic regression model
baseline_logit = smf.logit(
    formula='rec ~ rel + fun + clar + home + Q("class") + fac',
    data=learning_model_df
).fit()

# display model summary
print(baseline_logit.summary())

Optimization terminated successfully.
         Current function value: 0.530618
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                    rec   No. Observations:                 4120
Model:                          Logit   Df Residuals:                     4113
Method:                           MLE   Df Model:                            6
Date:                Mon, 25 May 2026   Pseudo R-squ.:                  0.2222
Time:                        20:58:45   Log-Likelihood:                -2186.1
converged:                       True   LL-Null:                       -2810.7
Covariance Type:            nonrobust   LLR p-value:                1.079e-266
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -7.0773      0.293    -24.164      0.000      -7.651      -6.503
rel            1.0016      0.

In [21]:
# odds ratios for baseline logistic regression
np.exp(baseline_logit.params)

,0
Intercept,0.000844
rel,2.722600
fun,0.910900
clar,1.697966
home,0.776924
"Q(""class"")",1.619137
fac,1.440333


**6. Baseline Logistic Diagnostics**

The diagnostics should be framed as the following:

* Technical diagnostics — did the model fit properly?
* Known limitation — independence is violated because of repeated participant records.

**Model convergence**
* This means the optimization algorithm successfully found stable coefficient estimates. So the model did not fail numerically. That is a good technical sign.

In [22]:
# confirm model convergence
print("Model converged:", baseline_logit.mle_retvals["converged"])

Model converged: True


Complete-case outcome distribution:

* rec = 0: about 57.4%
* rec = 1: about 42.6%

This is not severely imbalanced. We have enough examples of both recommendation and non-recommendation outcomes to fit a logistic model.

In [23]:
# outcome balance in complete-case modeling data
learning_model_df["rec"].value_counts(normalize=True)

,proportion
rec,
0,0.573786
1,0.426214


The **VIF** values are extremely high. This indicates substantial multicollinearity among the feedback predictors. That makes sense because people who rate one aspect of a program highly often rate other aspects highly too.

Practical implication:

* The model may still predict recommendation well, but individual coefficient estimates may be unstable or hard to interpret separately.

* So we should be cautious saying, for example, “clarity independently causes recommendation,” because clarity is strongly entangled with the overall positive evaluation pattern.

In [24]:
# VIF check for baseline logistic predictors
X_vif = learning_model_df[["rel", "fun", "clar", "home", "class", "fac"]].astype(float)

vif_df = pd.DataFrame({
    "variable": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

vif_df

,variable,VIF
0,rel,26.249909
1,fun,30.964097
2,clar,35.887145
3,home,27.217715
4,class,23.259305
5,fac,15.157962


The predicted probabilities range from about:

* minimum: 0.002
* median: 0.389
* maximum: 0.969

This shows the model produces a wide range of recommendation probabilities. It is not assigning everyone nearly the same probability, which is useful.



In [25]:
# fitted probabilities from baseline logistic model
learning_model_df["predicted_prob_baseline"] = baseline_logit.predict(learning_model_df)

learning_model_df["predicted_prob_baseline"].describe()

,predicted_prob_baseline
count,4120.000000
mean,0.426214
std,0.257804
min,0.002341
25%,0.209667
50%,0.388941
75%,0.638523
max,0.968588


Near-perfect probabilities

* 20 cases below 0.01
* 0 cases above 0.99

So there is a small number of very low predicted probabilities, but no evidence of complete separation or widespread near-perfect prediction. This is not a major concern.

In [26]:
# check for near-perfect predicted probabilities
print("Predicted probabilities near 0:")
print((learning_model_df["predicted_prob_baseline"] < 0.01).sum())

print("\nPredicted probabilities near 1:")
print((learning_model_df["predicted_prob_baseline"] > 0.99).sum())

Predicted probabilities near 0:
20

Predicted probabilities near 1:
0


ROC-AUC: 0.8075

That is fairly strong discrimination. It means the model is reasonably good at ranking feedback records by likelihood of recommendation.


In [27]:
# ROC-AUC for baseline logistic model
auc = roc_auc_score(
    learning_model_df["rec"],
    learning_model_df["predicted_prob_baseline"]
)

print(f"ROC-AUC: {auc:.4f}")

ROC-AUC: 0.8075


**Overall Diagnostic summary**

The baseline logistic model fits successfully and has good predictive discrimination. However, it has two major interpretive limitations: first, it ignores repeated observations from the same participant; second, the feedback predictors are highly collinear. Therefore, it is useful as a baseline model, but we should be cautious about treating the individual coefficients as fully independent effects.

**7. Hierarchical Logistic Regression**

Purpose:

* Estimate the effects of learning-experience ratings while accounting for participant-level differences in baseline tendency to recommend.

Plain meaning:

* Some participants may generally be generous reviewers, while others may be more critical. The random intercept allows each participant to have their own baseline recommendation tendency.

The hierarchical model uses the same complete-case sample as the baseline model:

* original rows: 4,974
* model rows: 4,120
* rows dropped: 854
* unique participants retained: 316

So we are comparing the baseline and hierarchical models on the same complete-case structure.

In [28]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

# add idcode back into complete-case modeling dataframe
hier_model_vars = ["idcode", "rec", "rel", "fun", "clar", "home", "class", "fac"]

learning_hier_df = learning_df[hier_model_vars].dropna().copy()

print("Original rows:", learning_df.shape[0])
print("Hierarchical model rows:", learning_hier_df.shape[0])
print("Rows dropped:", learning_df.shape[0] - learning_hier_df.shape[0])
print("Unique participants:", learning_hier_df["idcode"].nunique())

Original rows: 4974
Hierarchical model rows: 4120
Rows dropped: 854
Unique participants: 316


Fixed effects: Type = M

Rows marked M are the main/fixed-effect predictors. These are interpreted similarly to logistic regression coefficients: positive values mean higher log-odds of recommending the program.



Random effect: Type = V

The idcode row is the participant-level random intercept variance component.

The output says:

* posterior mean for idcode: about 0.7742
* this is modeled as a log standard deviation
* the corresponding standard deviation is shown around 2.169

That tells us there is meaningful participant-level variation in baseline recommendation tendency.

In plain English:

* Some participants are much more likely to recommend programs in general, while others are much less likely, even after accounting for their ratings of relevance, fun, clarity, homework, class quality, and faculty.

That confirms why the hierarchical model matters.

In [29]:
# hierarchical logistic regression with participant-level random intercept
hier_logit = BinomialBayesMixedGLM.from_formula(
    formula='rec ~ rel + fun + clar + home + Q("class") + fac',
    vc_formulas={
        "idcode": "0 + C(idcode)"
    },
    data=learning_hier_df
)

hier_logit_results = hier_logit.fit_vb()

print(hier_logit_results.summary())

               Binomial Mixed GLM Results
           Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
---------------------------------------------------------
Intercept     M   -14.7302   0.0460                      
rel           M     1.5457   0.0128                      
fun           M     0.2278   0.0117                      
clar          M     0.8736   0.0124                      
home          M     0.0183   0.0122                      
Q("class")    M     0.7395   0.0139                      
fac           M     0.8043   0.0143                      
idcode        V     0.7742   0.0397 2.169   2.003   2.348
Parameter types are mean structure (M) and variance
structure (V)
Variance parameters are modeled as log standard
deviations


**8. Compare Baseline vs Hierarchical Model**

Compare:

* coefficient direction
* coefficient magnitude
* statistical significance
* odds-ratio interpretation
* whether conclusions change after accounting for participant clustering

Key question:

*Do the same predictors remain important once participant-level differences are modeled?

**9. Interpretation Framework**

Final interpretation should answer:

* Which feedback dimension is most associated with recommendation?
* Are relevance, fun, clarity, homework/applicability, class quality, or facilities meaningful predictors?
* Does accounting for participant-level clustering change the conclusions?
* Is recommendation mostly driven by program qualities or by participant-level rating tendencies?

Participant-level random effect


Do participants differ in their baseline tendency to recommend?

* Yes. The participant-level random intercept is meaningful.

* Some participants are generally more likely to recommend programs than others, even after accounting for their ratings.

So the hierarchical model is telling us that program feedback matters, but participant-level tendencies also matter.



**10. Final Conclusions and Recommendations**

The hierarchical logistic model accounts for repeated feedback records from the same participant by including a participant-level random intercept. The fixed effects show that relevance remains the strongest positive predictor of recommendation, while clarity, class quality, and faculty/facilities also have positive associations. The random intercept for `idcode` indicates substantial variation across participants in their baseline tendency to recommend programs. This means that some participants are generally more likely or less likely to recommend programs, independent of their ratings on the program-quality dimensions. Therefore, the hierarchical model supports the broad findings from the baseline logistic model but provides a more appropriate structure because it accounts for repeated observations within participants.